<a href="https://colab.research.google.com/github/devansh98185/CSET343/blob/main/lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")


NumPy version: 2.1.3
Pandas version: 2.2.3
TensorFlow version: 2.21.0
Libraries imported successfully!


In [ ]:

# Load the Pima Indians Diabetes dataset

url = (
    "https://raw.githubusercontent.com/"
    "jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
)

columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df = pd.read_csv(url, header=None, names=columns)

# Display the first five rows
print("First five rows of the dataset:")
display(df.head())

# Check the dataset dimensions
print("\nDataset shape:", df.shape)

# Check column data types and non-null counts
print("\nDataset information:")
df.info()

# Check the target distribution
print("\nOutcome distribution:")
print(df["Outcome"].value_counts())


First five rows of the dataset:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Dataset shape: (768, 9)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Outcome distribution:
Outcome
0    500
1    268
Name: count, dtype: int64


In [ ]:

# 1. Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

# 2. Check for duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

# 3. Generate descriptive statistics
print("\nDescriptive statistics:")
display(df.describe().T)

# 4. Check the number of zeros in each column
print("\nNumber of zero values in each column:")
print((df == 0).sum())


Missing values in each column:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Number of duplicate rows: 0

Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00



Number of zero values in each column:
Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64


In [ ]:

# Identify columns where zero represents a missing measurement
zero_as_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

# Replace zero values with NaN in these columns
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

# Check missing values after replacement
print("Missing values after replacing zeros:")
print(df.isnull().sum())

# Display the first five rows
print("\nUpdated dataset:")
display(df.head())


Missing values after replacing zeros:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

Updated dataset:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,NaN,33.6,0.627,50,1
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31,0
2,8,183.0,64.0,NaN,NaN,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


In [ ]:

X = df.drop("Outcome", axis=1)
y = df["Outcome"].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (768, 8)
Target shape: (768,)


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


Training set: (614, 8)
Testing set: (154, 8)


In [ ]:

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# Learn medians from training data and transform both sets
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Missing values in training data:",
      np.isnan(X_train_imputed).sum())

print("Missing values in testing data:",
      np.isnan(X_test_imputed).sum())


Missing values in training data: 0
Missing values in testing data: 0


In [ ]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

print("\nMean of first 5 scaled training features:")
print(np.round(X_train_scaled.mean(axis=0)[:5], 3))

print("\nStandard deviation of first 5 scaled training features:")
print(np.round(X_train_scaled.std(axis=0)[:5], 3))


Training data shape: (614, 8)
Testing data shape: (154, 8)

Mean of first 5 scaled training features:
[-0. -0.  0. -0. -0.]

Standard deviation of first 5 scaled training features:
[1. 1. 1. 1. 1.]


In [ ]:

def build_model(
    input_dim=8,
    neurons=16,
    activation="relu",
    initializer="glorot_uniform",
    dropout_rate=0.2,
    learning_rate=0.001
):
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),

        layers.Dense(
            units=neurons,
            activation=activation,
            kernel_initializer=initializer
        ),

        layers.Dropout(dropout_rate),

        layers.Dense(
            units=1,
            activation="sigmoid"
        )
    ])

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


# Create a sample model to verify the architecture
sample_model = build_model(input_dim=X_train_scaled.shape[1])

sample_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161 (644.00 B)

 Trainable params: 161 (644.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:

from sklearn.base import BaseEstimator, ClassifierMixin

class KerasMLPClassifier(ClassifierMixin, BaseEstimator):

    def __init__(
        self,
        neurons=16,
        activation="relu",
        initializer="glorot_uniform",
        dropout_rate=0.2,
        learning_rate=0.001,
        batch_size=32,
        epochs=20,
        random_state=42
    ):
        self.neurons = neurons
        self.activation = activation
        self.initializer = initializer
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.random_state = random_state

    def fit(self, X, y):
        keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)

        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int32)

        self.classes_ = np.unique(y)

        self.model_ = build_model(
            input_dim=X.shape[1],
            neurons=self.neurons,
            activation=self.activation,
            initializer=self.initializer,
            dropout_rate=self.dropout_rate,
            learning_rate=self.learning_rate
        )

        self.model_.fit(
            X,
            y,
            batch_size=self.batch_size,
            epochs=self.epochs,
            verbose=0
        )

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)

        probabilities = self.model_.predict(
            X, verbose=0
        ).ravel()

        return np.column_stack([
            1 - probabilities,
            probabilities
        ])

    def predict(self, X):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= 0.5).astype(int)


In [ ]:

# Create the complete pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KerasMLPClassifier())
])

# Test the pipeline on a small subset first
pipeline.fit(X_train.iloc[:100], y_train.iloc[:100])

# Make predictions on a small test subset
test_predictions = pipeline.predict(X_test.iloc[:20])

print("Pipeline test successful!")
print("Predictions:", test_predictions)
print("Actual values:", y_test.iloc[:20].to_numpy())


Pipeline test successful!
Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0]
Actual values: [0 0 0 1 0 0 1 1 0 0 0 1 0 0 1 1 0 0 0 0]


In [ ]:

param_grids = {
    "Batch Size": {
        "model__batch_size": [16, 32]
    },

    "Training Epochs": {
        "model__epochs": [20, 40]
    },

    "Learning Rate": {
        "model__learning_rate": [0.001, 0.01]
    },

    "Weight Initializer": {
        "model__initializer": ["glorot_uniform", "he_normal"]
    },

    "Activation Function": {
        "model__activation": ["relu", "tanh"]
    },

    "Dropout Rate": {
        "model__dropout_rate": [0.0, 0.3]
    },

    "Hidden Neurons": {
        "model__neurons": [16, 32]
    }
}

print("Number of hyperparameters:", len(param_grids))

for name, grid in param_grids.items():
    print(f"{name}: {list(grid.values())[0]}")


Number of hyperparameters: 7
Batch Size: [16, 32]
Training Epochs: [20, 40]
Learning Rate: [0.001, 0.01]
Weight Initializer: ['glorot_uniform', 'he_normal']
Activation Function: ['relu', 'tanh']
Dropout Rate: [0.0, 0.3]
Hidden Neurons: [16, 32]


In [ ]:

# Tune batch size using GridSearchCV

batch_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Batch Size"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

batch_grid.fit(X_train, y_train)

print("\nBest batch size:", batch_grid.best_params_)
print("Best cross-validation accuracy:",
      round(batch_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(batch_grid.cv_results_)[
    ["param_model__batch_size", "mean_test_score", "std_test_score", "rank_test_score"]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits



Best batch size: {'model__batch_size': 16}
Best cross-validation accuracy: 0.7508

Results for each candidate:


,param_model__batch_size,mean_test_score,std_test_score,rank_test_score
0,16,0.750781,0.016471,1
1,32,0.736139,0.010961,2


In [ ]:

# Tune the number of training epochs

epochs_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Training Epochs"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

epochs_grid.fit(X_train, y_train)

print("\nBest number of epochs:", epochs_grid.best_params_)
print("Best cross-validation accuracy:",
      round(epochs_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(epochs_grid.cv_results_)[
    [
        "param_model__epochs",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best number of epochs: {'model__epochs': 40}
Best cross-validation accuracy: 0.7671

Results for each candidate:


,param_model__epochs,mean_test_score,std_test_score,rank_test_score
0,20,0.736139,0.010961,2
1,40,0.767089,0.009486,1


In [ ]:

# Tune the learning rate

learning_rate_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Learning Rate"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

learning_rate_grid.fit(X_train, y_train)

print("\nBest learning rate:", learning_rate_grid.best_params_)
print("Best cross-validation accuracy:",
      round(learning_rate_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(learning_rate_grid.cv_results_)[
    [
        "param_model__learning_rate",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best learning rate: {'model__learning_rate': 0.01}
Best cross-validation accuracy: 0.7737

Results for each candidate:


,param_model__learning_rate,mean_test_score,std_test_score,rank_test_score
0,0.001,0.736139,0.010961,2
1,0.010,0.773665,0.021487,1


In [ ]:

# Tune the weight initializer

initializer_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Weight Initializer"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

initializer_grid.fit(X_train, y_train)

print("\nBest weight initializer:", initializer_grid.best_params_)
print("Best cross-validation accuracy:",
      round(initializer_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(initializer_grid.cv_results_)[
    [
        "param_model__initializer",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best weight initializer: {'model__initializer': 'he_normal'}
Best cross-validation accuracy: 0.741

Results for each candidate:


,param_model__initializer,mean_test_score,std_test_score,rank_test_score
0,glorot_uniform,0.736139,0.010961,2
1,he_normal,0.741049,0.018150,1


In [ ]:

# Tune the activation function

activation_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Activation Function"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

activation_grid.fit(X_train, y_train)

print("\nBest activation function:", activation_grid.best_params_)
print("Best cross-validation accuracy:",
      round(activation_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(activation_grid.cv_results_)[
    [
        "param_model__activation",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best activation function: {'model__activation': 'tanh'}
Best cross-validation accuracy: 0.759

Results for each candidate:


,param_model__activation,mean_test_score,std_test_score,rank_test_score
0,relu,0.736139,0.010961,2
1,tanh,0.758991,0.017540,1


In [ ]:

# Tune the dropout rate

dropout_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Dropout Rate"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

dropout_grid.fit(X_train, y_train)

print("\nBest dropout rate:", dropout_grid.best_params_)
print("Best cross-validation accuracy:",
      round(dropout_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(dropout_grid.cv_results_)[
    [
        "param_model__dropout_rate",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best dropout rate: {'model__dropout_rate': 0.0}
Best cross-validation accuracy: 0.741

Results for each candidate:


,param_model__dropout_rate,mean_test_score,std_test_score,rank_test_score
0,0.0,0.741049,0.010361,1
1,0.3,0.724733,0.015494,2


In [ ]:

# Tune the number of neurons in the hidden layer

neurons_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grids["Hidden Neurons"],
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
    error_score="raise",
    verbose=1
)

neurons_grid.fit(X_train, y_train)

print("\nBest number of hidden neurons:", neurons_grid.best_params_)
print("Best cross-validation accuracy:",
      round(neurons_grid.best_score_, 4))

print("\nResults for each candidate:")
display(pd.DataFrame(neurons_grid.cv_results_)[
    [
        "param_model__neurons",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
])


Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best number of hidden neurons: {'model__neurons': 32}
Best cross-validation accuracy: 0.7622

Results for each candidate:


,param_model__neurons,mean_test_score,std_test_score,rank_test_score
0,16,0.736139,0.010961,2
1,32,0.762211,0.002620,1


In [ ]:

# Combine the best results from all seven searches

all_results = pd.DataFrame([
    {
        "Hyperparameter": "Batch Size",
        "Best Value": batch_grid.best_params_["model__batch_size"],
        "Best CV Accuracy": batch_grid.best_score_
    },
    {
        "Hyperparameter": "Epochs",
        "Best Value": epochs_grid.best_params_["model__epochs"],
        "Best CV Accuracy": epochs_grid.best_score_
    },
    {
        "Hyperparameter": "Learning Rate",
        "Best Value": learning_rate_grid.best_params_["model__learning_rate"],
        "Best CV Accuracy": learning_rate_grid.best_score_
    },
    {
        "Hyperparameter": "Weight Initializer",
        "Best Value": initializer_grid.best_params_["model__initializer"],
        "Best CV Accuracy": initializer_grid.best_score_
    },
    {
        "Hyperparameter": "Activation Function",
        "Best Value": activation_grid.best_params_["model__activation"],
        "Best CV Accuracy": activation_grid.best_score_
    },
    {
        "Hyperparameter": "Dropout Rate",
        "Best Value": dropout_grid.best_params_["model__dropout_rate"],
        "Best CV Accuracy": dropout_grid.best_score_
    },
    {
        "Hyperparameter": "Hidden Neurons",
        "Best Value": neurons_grid.best_params_["model__neurons"],
        "Best CV Accuracy": neurons_grid.best_score_
    }
])

display(all_results.round(4))

all_results.to_csv("lab7_hyperparameter_results.csv", index=False)
print("Results saved successfully.")


,Hyperparameter,Best Value,Best CV Accuracy
0,Batch Size,16,0.7508
1,Epochs,40,0.7671
2,Learning Rate,0.01,0.7737
3,Weight Initializer,he_normal,0.7410
4,Activation Function,tanh,0.7590
5,Dropout Rate,0.0,0.7410
6,Hidden Neurons,32,0.7622


Results saved successfully.
